데이터 출처 : [카드 소비 데이터](https://data.gg.go.kr/portal/data/service/selectServicePage.do?page=1&rows=10&sortColumn=&sortDirection=&infId=7Y02TF04H1WUB55Q4IZL35052374&infSeq=1&order=)

In [ ]:
import pandas as pd
import numpy as np
import zipfile
import os

In [ ]:
data_dir = "data"

# 월별 zip 파일 매핑
zip_files = {
    "2507": "카드소비 데이터_202507.zip",
    "2508": "카드소비 데이터_202508.zip",
    "2509": "카드소비 데이터_202509.zip",
    "2510": "카드소비 데이터_202510.zip",
    "2511": "카드소비 데이터_202511.zip",
    "2512": "카드소비 데이터_202512.zip",
}

# 월별로 전체 시 데이터를 합쳐서 하나의 DataFrame으로 생성
for month, zipname in zip_files.items():
    zip_path = os.path.join(data_dir, zipname)
    monthly_dfs = []

    with zipfile.ZipFile(zip_path) as zf:
        for filename in sorted(zf.namelist()):
            with zf.open(filename) as f:
                df_city = pd.read_csv(f)
                # 어느 시 데이터인지 컬럼 추가
                city_name = filename.split("_")[-1].replace(".csv", "")
                df_city["시군구"] = city_name
                monthly_dfs.append(df_city)

    # 해당 월의 전체 시 데이터 합치기
    combined = pd.concat(monthly_dfs, ignore_index=True)
    globals()[f"df_{month}"] = combined
    print(f"✅ df_{month} 생성 완료 → shape: {combined.shape}")

In [ ]:
# 생성된 월별 DataFrame 확인
for month in ["2507", "2508", "2509", "2510", "2511", "2512"]:
    df = globals()[f"df_{month}"]
    print(f"df_{month}: {df.shape} | 시군구 목록: {df['시군구'].unique().tolist()}")

In [ ]:
# 예시: 7월 데이터 미리보기
df_2507.head()

## EDA

In [ ]:
# 대분류 columns 및 항목별 counts
df_2507['card_tpbuz_nm_1'].value_counts()

In [ ]:
# 음식 업종의 소분류
df_2507[df_2507['card_tpbuz_nm_1']=='음식']['card_tpbuz_nm_2'].value_counts()

In [ ]:
# 소매/유통 업종의 소분류
df_2507[df_2507['card_tpbuz_nm_1']=='소매/유통']['card_tpbuz_nm_2'].value_counts()

In [ ]:
# 생활 서비스 업종의 소분류
df_2507[df_2507['card_tpbuz_nm_1']=='생활서비스']['card_tpbuz_nm_2'].value_counts()

In [ ]:
# 의료/건강 업종의 소분류
df_2507[df_2507['card_tpbuz_nm_1']=='의료/건강']['card_tpbuz_nm_2'].value_counts()

In [ ]:
# 여가 오락 업종의 소분류
df_2507[df_2507['card_tpbuz_nm_1']=='여가/오락']['card_tpbuz_nm_2'].value_counts()

In [ ]:
# 학문/교육 업종의 소분류
df_2507[df_2507['card_tpbuz_nm_1']=='학문/교육']['card_tpbuz_nm_2'].value_counts()

In [ ]:
# 공공/기업/단체 업종의 소분류
df_2507[df_2507['card_tpbuz_nm_1']=='공공/기업/단체']['card_tpbuz_nm_2'].value_counts()

In [ ]:
# 미디어 통신 업종의 소분류
df_2507[df_2507['card_tpbuz_nm_1']=='미디어/통신']['card_tpbuz_nm_2'].value_counts()

In [ ]:
# 공연 전시 업종의 소분류
df_2507[df_2507['card_tpbuz_nm_1']=='공연/전시']['card_tpbuz_nm_2'].value_counts()

---

## box plot

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False


In [ ]:

# ─────────────────────────────────────────────────────────────
# 분석 대상 DataFrame 설정 (원하는 월로 변경 가능)
# df_2507 / df_2508 / df_2509 / df_2510 / df_2511 / df_2512
# ─────────────────────────────────────────────────────────────
df_target = df_2507.copy()

# ══════════════════════════════════════════════════════════════
# ★ 건당 가격 컬럼 생성 : amt / cnt
#   cnt = 0 인 행은 NaN 처리 후 제거
# ══════════════════════════════════════════════════════════════
df_target['건당가격'] = df_target['amt'] / df_target['cnt']
df_target = df_target[df_target['cnt'] > 0].copy()   # cnt=0 행 제거

print(f"▶ 건당가격 컬럼 생성 완료")
print(f"  - 전체 행수     : {len(df_target):,}")
print(f"  - 건당가격 최솟값: {df_target['건당가격'].min():,.0f}원")
print(f"  - 건당가격 최댓값: {df_target['건당가격'].max():,.0f}원")
print(f"  - 건당가격 평균  : {df_target['건당가격'].mean():,.0f}원")
print(f"  - 건당가격 중앙값: {df_target['건당가격'].median():,.0f}원")
display(df_target[['ta_ymd','card_tpbuz_nm_1','card_tpbuz_nm_2','amt','cnt','건당가격']].head(5))

# ══════════════════════════════════════════════════════════════
# 1) card_tpbuz_nm_1 (대분류) 기준 건당가격 통계 요약
# ══════════════════════════════════════════════════════════════
stats_nm1 = df_target.groupby('card_tpbuz_nm_1')['건당가격'].agg(
    건수='count',
    평균='mean',
    중앙값='median',
    표준편차='std',
    최솟값='min',
    최댓값='max',
    Q1=lambda x: x.quantile(0.25),
    Q3=lambda x: x.quantile(0.75),
).round(0)
stats_nm1['IQR'] = stats_nm1['Q3'] - stats_nm1['Q1']
stats_nm1['이상치_상한'] = (stats_nm1['Q3'] + 1.5 * stats_nm1['IQR']).round(0)
stats_nm1['이상치_하한'] = (stats_nm1['Q1'] - 1.5 * stats_nm1['IQR']).round(0)

print("\n" + "=" * 65)
print("  [nm_1] 업종 대분류 기준  건당가격  통계 요약")
print("=" * 65)
display(stats_nm1)

# ══════════════════════════════════════════════════════════════
# 2) card_tpbuz_nm_1 기준 건당가격 Boxplot
# ══════════════════════════════════════════════════════════════
order_nm1 = stats_nm1.sort_values('중앙값', ascending=False).index.tolist()
groups_nm1 = [df_target[df_target['card_tpbuz_nm_1'] == nm]['건당가격'].values
              for nm in order_nm1]

fig, ax = plt.subplots(figsize=(15, 6))
ax.boxplot(
    groups_nm1, labels=order_nm1, patch_artist=True,
    showmeans=True, meanline=True,
    boxprops=dict(facecolor='#AED6F1', color='#2980B9'),
    medianprops=dict(color='#E74C3C', linewidth=2.5),
    meanprops=dict(color='#27AE60', linewidth=2, linestyle='--'),
    whiskerprops=dict(color='#2980B9', linewidth=1.2),
    capprops=dict(color='#2980B9', linewidth=1.5),
    flierprops=dict(marker='o', markerfacecolor='#E74C3C',
                    markersize=3, alpha=0.35, markeredgewidth=0)
)
ax.set_title(
    '업종 대분류(card_tpbuz_nm_1) 기준  건당가격  Boxplot\n'
    '─ 빨간선=중앙값  /  초록점선=평균  /  빨간점=이상치',
    fontsize=13, pad=12
)
ax.set_xlabel('업종 대분류', fontsize=11)
ax.set_ylabel('건당가격 (원)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(rotation=30, ha='right', fontsize=10)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════
# 3) card_tpbuz_nm_2 (소분류) 기준 건당가격 통계 요약
# ══════════════════════════════════════════════════════════════
stats_nm2 = df_target.groupby('card_tpbuz_nm_2')['건당가격'].agg(
    건수='count',
    평균='mean',
    중앙값='median',
    표준편차='std',
    최솟값='min',
    최댓값='max',
    Q1=lambda x: x.quantile(0.25),
    Q3=lambda x: x.quantile(0.75),
).round(0).sort_values('평균', ascending=False)
stats_nm2['IQR'] = stats_nm2['Q3'] - stats_nm2['Q1']
stats_nm2['이상치_상한'] = (stats_nm2['Q3'] + 1.5 * stats_nm2['IQR']).round(0)
stats_nm2['이상치_하한'] = (stats_nm2['Q1'] - 1.5 * stats_nm2['IQR']).round(0)

print("\n" + "=" * 65)
print("  [nm_2] 업종 소분류 기준  건당가격  통계 요약 (평균 내림차순)")
print("=" * 65)
display(stats_nm2)

# ══════════════════════════════════════════════════════════════
# 4) card_tpbuz_nm_2 기준 건당가격 Boxplot (거래건수 상위 20개)
# ══════════════════════════════════════════════════════════════
top20_nm2 = stats_nm2.nlargest(20, '건수').index.tolist()
groups_nm2 = [df_target[df_target['card_tpbuz_nm_2'] == nm]['건당가격'].values
              for nm in top20_nm2]

fig, ax = plt.subplots(figsize=(22, 7))
ax.boxplot(
    groups_nm2, labels=top20_nm2, patch_artist=True,
    showmeans=True, meanline=True,
    boxprops=dict(facecolor='#A9DFBF', color='#1E8449'),
    medianprops=dict(color='#E74C3C', linewidth=2.5),
    meanprops=dict(color='#8E44AD', linewidth=2, linestyle='--'),
    whiskerprops=dict(color='#1E8449', linewidth=1.2),
    capprops=dict(color='#1E8449', linewidth=1.5),
    flierprops=dict(marker='o', markerfacecolor='#E74C3C',
                    markersize=3, alpha=0.3, markeredgewidth=0)
)
ax.set_title(
    '업종 소분류(card_tpbuz_nm_2) 거래건수 상위 20개  건당가격  Boxplot\n'
    '─ 빨간선=중앙값  /  보라점선=평균  /  빨간점=이상치',
    fontsize=13, pad=12
)
ax.set_xlabel('업종 소분류', fontsize=11)
ax.set_ylabel('건당가격 (원)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(rotation=40, ha='right', fontsize=10)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════
# 5) 대분류 + 소분류 조합 이상치 비율 요약
# ══════════════════════════════════════════════════════════════
def count_outliers(x):
    Q1, Q3 = x.quantile(0.25), x.quantile(0.75)
    IQR = Q3 - Q1
    return int(((x < Q1 - 1.5 * IQR) | (x > Q3 + 1.5 * IQR)).sum())

outlier_summary = df_target.groupby(['card_tpbuz_nm_1', 'card_tpbuz_nm_2'])['건당가격'].agg(
    건수='count',
    평균='mean',
    중앙값='median',
    이상치건수=count_outliers
).round(0)
outlier_summary['이상치비율(%)'] = (
    outlier_summary['이상치건수'] / outlier_summary['건수'] * 100
).round(2)
outlier_summary = outlier_summary.sort_values('이상치비율(%)', ascending=False)

print("\n" + "=" * 65)
print("  업종 대/소분류 조합별  건당가격  이상치 비율 상위 30개")
print("=" * 65)
display(outlier_summary.head(30))


---

## 이상치 제거

In [ ]:
# ─────────────────────────────────────────────────────────────
# 분석 대상 DataFrame 설정 (원하는 월로 변경 가능)
# ─────────────────────────────────────────────────────────────
df_target = df_2507.copy()

# ══════════════════════════════════════════════════════════════
# ① 건당가격 컬럼 생성
# ══════════════════════════════════════════════════════════════
df_target = df_target[df_target['cnt'] > 0].copy()
df_target['건당가격'] = df_target['amt'] / df_target['cnt']

print(f"[이상치 제거 전] 전체 행수: {len(df_target):,}")

# ══════════════════════════════════════════════════════════════
# ② 업종 소분류(nm_2) 그룹별 IQR 이상치 제거
#    → 업종마다 가격대가 다르기 때문에 그룹 내에서 각각 제거
# ══════════════════════════════════════════════════════════════
def remove_outliers_iqr(group):
    Q1 = group['건당가격'].quantile(0.25)
    Q3 = group['건당가격'].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return group[(group['건당가격'] >= lower) & (group['건당가격'] <= upper)]

df_clean = df_target.groupby('card_tpbuz_nm_2', group_keys=False).apply(remove_outliers_iqr)
df_clean = df_clean.reset_index(drop=True)

removed = len(df_target) - len(df_clean)
print(f"[이상치 제거 후] 전체 행수: {len(df_clean):,}")
print(f"  제거된 행수   : {removed:,} ({removed/len(df_target)*100:.2f}%)")

# ══════════════════════════════════════════════════════════════
# ③ card_tpbuz_nm_1 (대분류) 기준 건당가격 통계 요약
# ══════════════════════════════════════════════════════════════
stats_nm1 = df_clean.groupby('card_tpbuz_nm_1')['건당가격'].agg(
    건수='count',
    평균='mean',
    중앙값='median',
    표준편차='std',
    최솟값='min',
    최댓값='max',
    Q1=lambda x: x.quantile(0.25),
    Q3=lambda x: x.quantile(0.75),
).round(0)
stats_nm1['IQR'] = stats_nm1['Q3'] - stats_nm1['Q1']

print("\n" + "=" * 65)
print("  [nm_1] 업종 대분류 기준  건당가격  통계 (이상치 제거 후)")
print("=" * 65)
display(stats_nm1)

# ══════════════════════════════════════════════════════════════
# ④ card_tpbuz_nm_1 기준 Boxplot (이상치 제거 후)
# ══════════════════════════════════════════════════════════════
order_nm1 = stats_nm1.sort_values('중앙값', ascending=False).index.tolist()
groups_nm1 = [df_clean[df_clean['card_tpbuz_nm_1'] == nm]['건당가격'].values
              for nm in order_nm1]

fig, ax = plt.subplots(figsize=(15, 6))
ax.boxplot(
    groups_nm1, labels=order_nm1, patch_artist=True,
    showmeans=True, meanline=True,
    boxprops=dict(facecolor='#AED6F1', color='#2980B9'),
    medianprops=dict(color='#E74C3C', linewidth=2.5),
    meanprops=dict(color='#27AE60', linewidth=2, linestyle='--'),
    whiskerprops=dict(color='#2980B9', linewidth=1.2),
    capprops=dict(color='#2980B9', linewidth=1.5),
    flierprops=dict(marker='o', markerfacecolor='#E74C3C',
                    markersize=3, alpha=0.35, markeredgewidth=0)
)
ax.set_title(
    '업종 대분류(nm_1) 기준  건당가격  Boxplot  ✅ 이상치 제거 후\n'
    '─ 빨간선=중앙값  /  초록점선=평균',
    fontsize=13, pad=12
)
ax.set_xlabel('업종 대분류', fontsize=11)
ax.set_ylabel('건당가격 (원)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(rotation=30, ha='right', fontsize=10)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════
# ⑤ card_tpbuz_nm_2 (소분류) 기준 건당가격 통계 요약
# ══════════════════════════════════════════════════════════════
stats_nm2 = df_clean.groupby('card_tpbuz_nm_2')['건당가격'].agg(
    건수='count',
    평균='mean',
    중앙값='median',
    표준편차='std',
    최솟값='min',
    최댓값='max',
    Q1=lambda x: x.quantile(0.25),
    Q3=lambda x: x.quantile(0.75),
).round(0).sort_values('평균', ascending=False)
stats_nm2['IQR'] = stats_nm2['Q3'] - stats_nm2['Q1']

print("\n" + "=" * 65)
print("  [nm_2] 업종 소분류 기준  건당가격  통계 (이상치 제거 후, 평균 내림차순)")
print("=" * 65)
display(stats_nm2)

# ══════════════════════════════════════════════════════════════
# ⑥ card_tpbuz_nm_2 기준 Boxplot (거래건수 상위 20개, 이상치 제거 후)
# ══════════════════════════════════════════════════════════════
top20_nm2 = stats_nm2.nlargest(20, '건수').index.tolist()
groups_nm2 = [df_clean[df_clean['card_tpbuz_nm_2'] == nm]['건당가격'].values
              for nm in top20_nm2]

fig, ax = plt.subplots(figsize=(22, 7))
ax.boxplot(
    groups_nm2, labels=top20_nm2, patch_artist=True,
    showmeans=True, meanline=True,
    boxprops=dict(facecolor='#A9DFBF', color='#1E8449'),
    medianprops=dict(color='#E74C3C', linewidth=2.5),
    meanprops=dict(color='#8E44AD', linewidth=2, linestyle='--'),
    whiskerprops=dict(color='#1E8449', linewidth=1.2),
    capprops=dict(color='#1E8449', linewidth=1.5),
    flierprops=dict(marker='o', markerfacecolor='#E74C3C',
                    markersize=3, alpha=0.3, markeredgewidth=0)
)
ax.set_title(
    '업종 소분류(nm_2) 거래건수 상위 20개  건당가격  Boxplot  ✅ 이상치 제거 후\n'
    '─ 빨간선=중앙값  /  보라점선=평균',
    fontsize=13, pad=12
)
ax.set_xlabel('업종 소분류', fontsize=11)
ax.set_ylabel('건당가격 (원)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(rotation=40, ha='right', fontsize=10)
plt.tight_layout()
plt.show()


---

## 클러스터링

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
# ─────────────────────────────────────────────────────────────
# ① 데이터 로딩 (eda.ipynb와 동일한 방식)
# ─────────────────────────────────────────────────────────────
data_dir = "data"

zip_files = {
    "2507": "카드소비 데이터_202507.zip",
    "2508": "카드소비 데이터_202508.zip",
    "2509": "카드소비 데이터_202509.zip",
    "2510": "카드소비 데이터_202510.zip",
    "2511": "카드소비 데이터_202511.zip",
    "2512": "카드소비 데이터_202512.zip",
}

all_dfs = []
for month, zipname in zip_files.items():
    zip_path = os.path.join(data_dir, zipname)
    monthly_dfs = []

    with zipfile.ZipFile(zip_path) as zf:
        for filename in sorted(zf.namelist()):
            with zf.open(filename) as f:
                df_city = pd.read_csv(f)
                city_name = filename.split("_")[-1].replace(".csv", "")
                df_city["시군구"] = city_name
                monthly_dfs.append(df_city)

    combined = pd.concat(monthly_dfs, ignore_index=True)
    combined['month'] = month   # 월 컬럼 추가
    all_dfs.append(combined)
    print(f"✅ {month} 로드 완료 → shape: {combined.shape}")

# ─────────────────────────────────────────────────────────────
# ② 전체 통합 & 전처리
# ─────────────────────────────────────────────────────────────
df_all = pd.concat(all_dfs, ignore_index=True)
print(f"\n✅ 전체 합치기 완료 → shape: {df_all.shape}")

# ── 도시(시군구) 컬럼 제거 ───────────────────────────────────
df_all = df_all.drop(columns=['시군구'], errors='ignore')
print(f"   시군구 컬럼 제거 완료")

# ── cnt=0 제거 & 건당가격 생성 ───────────────────────────────
df_all = df_all[df_all['cnt'] > 0].copy()
df_all['건당가격'] = df_all['amt'] / df_all['cnt']
print(f"   cnt>0 필터 후 shape   : {df_all.shape}")

# ── 이상치 제거: card_tpbuz_nm_2 그룹별 IQR ─────────────────
def remove_outliers_iqr(group):
    Q1 = group['건당가격'].quantile(0.25)
    Q3 = group['건당가격'].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return group[(group['건당가격'] >= lower) & (group['건당가격'] <= upper)]

before = len(df_all)
df_all = df_all.groupby('card_tpbuz_nm_2', group_keys=False).apply(remove_outliers_iqr)
after  = len(df_all)
print(f"   이상치 제거 전: {before:,}행  →  후: {after:,}행  (제거: {before - after:,}행)")
print(f"   건당가격 평균          : {df_all['건당가격'].mean():,.0f}원")
print(f"   건당가격 중앙값        : {df_all['건당가격'].median():,.0f}원")

# ══════════════════════════════════════════════════════════════
# ③ 피처 엔지니어링: (age, sex) 단위로 소비자 프로파일 집계
# ══════════════════════════════════════════════════════════════

# 기본 집계: 건당가격 평균, 총 거래건수, 총 결제금액, 평균 시간대
base = df_all.groupby(['age', 'sex']).agg(
    평균_건당가격=('건당가격', 'mean'),
    총_거래건수=('cnt', 'sum'),
    총_결제금액=('amt', 'sum'),
    평균_시간대=('hour', 'mean'),
).reset_index()

# 업종 대분류(nm_1)별 결제비율 피벗
cat_pivot = df_all.groupby(['age', 'sex', 'card_tpbuz_nm_1'])['amt'].sum().reset_index()
cat_pivot = cat_pivot.pivot_table(
    index=['age', 'sex'],
    columns='card_tpbuz_nm_1',
    values='amt',
    fill_value=0
)
# 비율로 변환 (각 그룹 내 업종 비중)
cat_pivot = cat_pivot.div(cat_pivot.sum(axis=1), axis=0).round(4)
cat_pivot.columns = [f'업종비율_{c}' for c in cat_pivot.columns]
cat_pivot = cat_pivot.reset_index()

# 주말(day>=5) 비율
df_all['is_weekend'] = df_all['day'].apply(lambda x: 1 if x >= 5 else 0)
weekend = df_all.groupby(['age', 'sex'])['is_weekend'].mean().reset_index()
weekend.rename(columns={'is_weekend': '주말비율'}, inplace=True)

# 전체 합치기
profile = base.merge(cat_pivot, on=['age', 'sex']).merge(weekend, on=['age', 'sex'])

print(f"\n소비자 프로파일 shape: {profile.shape}")
print(f"컬럼 목록:\n{profile.columns.tolist()}")
print(profile.head())

# ══════════════════════════════════════════════════════════════
# ④ 피처 선택 & 스케일링
# ══════════════════════════════════════════════════════════════
feature_cols = [c for c in profile.columns if c not in ['age', 'sex']]
X = profile[feature_cols].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\n클러스터링 피처 수: {len(feature_cols)}개")
print(feature_cols)

# ══════════════════════════════════════════════════════════════
# ⑤ Elbow Method + Silhouette Score
# ══════════════════════════════════════════════════════════════
K_range = range(2, 12)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))
    print(f"  k={k:2d} | Inertia: {km.inertia_:10.1f} | Silhouette: {silhouette_score(X_scaled, labels):.4f}")

# ── 시각화 ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow Curve
axes[0].plot(list(K_range), inertias, 'bo-', linewidth=2, markersize=7)
axes[0].set_title('Elbow Method\n(Inertia)', fontsize=13)
axes[0].set_xlabel('클러스터 수 K')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].grid(linestyle='--', alpha=0.6)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Silhouette Score
axes[1].plot(list(K_range), silhouettes, 'rs-', linewidth=2, markersize=7)
best_k = list(K_range)[silhouettes.index(max(silhouettes))]
axes[1].axvline(x=best_k, color='orange', linestyle='--', linewidth=2,
                label=f'최적 K = {best_k}')
axes[1].set_title('Silhouette Score\n(높을수록 클러스터 분리가 잘 됨)', fontsize=13)
axes[1].set_xlabel('클러스터 수 K')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()
axes[1].grid(linestyle='--', alpha=0.6)

plt.suptitle('K-Means 최적 K 탐색 (소비자 유형 클러스터링, 7월~12월 전체)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"\n★ Silhouette 기준 최적 K = {best_k}")
print("  ※ Elbow 그래프에서 꺾이는 지점도 함께 참고하세요!")


In [ ]:
# ══════════════════════════════════════════════════════════════
# ⑥ best_k 로 실제 K-Means 클러스터링 실행
# ══════════════════════════════════════════════════════════════
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
profile['cluster'] = km_final.fit_predict(X_scaled)

print(f"\n[클러스터 분포]")
print(profile['cluster'].value_counts().sort_index())

# ── 클러스터별 원본 스케일 평균값 ────────────────────────────
cluster_means = profile.groupby('cluster')[feature_cols].mean()
print("\n[클러스터별 피처 평균값]")
print(cluster_means.T.to_string())

# ══════════════════════════════════════════════════════════════
# ⑦ 클러스터 중심 히트맵 (표준화된 값 → 어떤 피처가 두드러지는지)
# ══════════════════════════════════════════════════════════════
# 클러스터 중심값 (StandardScaler 기준)
centers_df = pd.DataFrame(
    km_final.cluster_centers_,
    columns=feature_cols,
    index=[f'Cluster {i}' for i in range(best_k)]
)

fig, ax = plt.subplots(figsize=(max(14, len(feature_cols) * 0.7), best_k * 1.2 + 2))
im = ax.imshow(centers_df.values, cmap='RdYlGn', aspect='auto')
ax.set_xticks(range(len(feature_cols)))
ax.set_xticklabels(feature_cols, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(best_k))
ax.set_yticklabels([f'Cluster {i}' for i in range(best_k)], fontsize=11)
plt.colorbar(im, ax=ax, label='표준화된 값 (+ 높음 / - 낮음)')
for i in range(best_k):
    for j in range(len(feature_cols)):
        ax.text(j, i, f'{centers_df.values[i, j]:.2f}',
                ha='center', va='center', fontsize=7,
                color='black')
ax.set_title('클러스터 중심값 히트맵\n(초록=높음, 빨강=낮음, 0=평균 수준)', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════
# ⑧ 피처 중요도: 클러스터 간 분산이 큰 피처 순위
# ══════════════════════════════════════════════════════════════
# 클러스터 중심값의 표준편차 → 클러스터 간 차이가 큰 피처일수록 영향력 큼
feature_importance = centers_df.std(axis=0).sort_values(ascending=False)
print("\n[클러스터 간 차이가 큰 피처 순위 (중요도 높은 순)]")
print(feature_importance.to_string())

fig, ax = plt.subplots(figsize=(12, max(5, len(feature_cols) * 0.4)))
colors = ['#E74C3C' if v > feature_importance.mean() else '#3498DB'
          for v in feature_importance.values]
ax.barh(feature_importance.index[::-1], feature_importance.values[::-1], color=colors[::-1])
ax.axvline(x=feature_importance.mean(), color='gray', linestyle='--', alpha=0.7,
           label=f'평균 중요도 = {feature_importance.mean():.3f}')
ax.set_title('피처 중요도 (클러스터 간 분산 기준)\n빨강=평균 초과 / 파랑=평균 이하', fontsize=13)
ax.set_xlabel('클러스터 중심값의 표준편차 (클러스터 간 차별화 정도)')
ax.legend()
ax.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════
# ⑨ 상위 중요 피처별 클러스터 평균 바플롯
# ══════════════════════════════════════════════════════════════
top_n = min(8, len(feature_cols))
top_features = feature_importance.head(top_n).index.tolist()

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
palette = plt.cm.Set2.colors

for idx, feat in enumerate(top_features):
    ax = axes[idx]
    vals = cluster_means[feat].values
    bars = ax.bar(
        [f'C{i}' for i in range(best_k)],
        vals,
        color=[palette[i % len(palette)] for i in range(best_k)],
        edgecolor='gray', linewidth=0.5
    )
    ax.set_title(feat, fontsize=10, pad=5)
    ax.set_ylabel('평균값', fontsize=8)
    ax.tick_params(axis='x', labelsize=9)
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    # 값 표시
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f'{v:,.1f}', ha='center', va='bottom', fontsize=8)

# 남은 subplot 숨기기
for idx in range(top_n, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle(f'상위 {top_n}개 주요 피처별 클러스터 평균 비교', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════
# ⑩ 클러스터별 age / sex 분포
# ══════════════════════════════════════════════════════════════
print("\n[클러스터별 age 분포]")
print(profile.groupby('cluster')['age'].value_counts().unstack(fill_value=0))

print("\n[클러스터별 sex 분포]")
print(profile.groupby('cluster')['sex'].value_counts().unstack(fill_value=0))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# age 분포
profile.groupby(['cluster', 'age']).size().unstack(fill_value=0).plot(
    kind='bar', ax=axes[0], colormap='tab20', edgecolor='gray', linewidth=0.5
)
axes[0].set_title('클러스터별 연령대(age) 분포', fontsize=12)
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('소비자 그룹 수')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='age', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

# sex 분포
profile.groupby(['cluster', 'sex']).size().unstack(fill_value=0).plot(
    kind='bar', ax=axes[1], colormap='Set1', edgecolor='gray', linewidth=0.5
)
axes[1].set_title('클러스터별 성별(sex) 분포', fontsize=12)
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('소비자 그룹 수')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='sex', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
axes[1].grid(axis='y', linestyle='--', alpha=0.5)

plt.suptitle('클러스터별 소비자 인구통계 분포', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 각 클러스터에 어떤 (age, sex) 조합이 들어있는지 출력
print(profile[['age', 'sex', 'cluster']].sort_values(['cluster', 'age']))

# 소비자 클러스터 5유형 프로파일

> **분석 개요**  
> 카드 소비 데이터(2025년 7월~12월) 기반 K-Means 클러스터링 결과  
> 이상치 제거(IQR, 업종 소분류 기준) → 소비 패턴 피처 집계 → K=5 최적 군집

---

## 🍽️ Cluster 0 — 외식·여가 중심 소비자

| 항목 | 내용 |
|------|------|
| **건당 평균** | 13,827원 |
| **주요 업종** | 음식(35%) · 소매/유통(38%) · 여가/오락(6%) |
| **주말 소비 비율** | 42.5% |

**특징**  
적당한 가격대로 자주 소비하며, 주말에 외식·카페·문화생활에 적극적으로 지갑을 연다.

> 💬 맛집과 놀이문화에 적극적인 소비자

---

## 🛒 Cluster 1 — 소액 쇼핑 중심 소비자

| 항목 | 내용 |
|------|------|
| **건당 평균** | 6,833원 (전 클러스터 최저) |
| **주요 업종** | 소매/유통(65%) |
| **주말 소비 비율** | 29.9% (전 클러스터 최저) |

**특징**  
소규모·소액 구매 중심이며 평일 소비 비율이 높다. 편의점·잡화 등 일상 쇼핑에 집중한다.

> 💬 편의점·잡화·일상 쇼핑에 집중하는 알뜰 소비자

---

## 🏠 Cluster 2 — 생활 전반 대량 소비자

| 항목 | 내용 |
|------|------|
| **건당 평균** | 29,374원 |
| **주요 업종** | 소매/유통(34%) · 생활서비스(23%) · 음식(24%) |
| **거래 규모** | 거래건수·결제금액 **전 클러스터 1위** |

**특징**  
생활 전반에 걸쳐 폭넓고 활발하게 소비하며, 생활서비스(수리·세탁·대형서비스 등) 지출 비중이 타 클러스터 대비 높다.

> 💬 가정과 생활을 위한 소비를 주도하는 핵심 소비층

---

## 🏥 Cluster 3 — 건강관리 중심 소비자

| 항목 | 내용 |
|------|------|
| **건당 평균** | 21,150원 |
| **주요 업종** | 소매/유통(36%) · **의료/건강(21.5%)** |
| **주말 소비 비율** | 39.2% |

**특징**  
병원·약국·건강기능식품 등 의료/건강 지출 비중이 타 클러스터 대비 압도적으로 높다. 평일 중심 소비 패턴을 보인다.

> 💬 건강 유지와 관리에 꾸준히 투자하는 소비자

---

## 📚 Cluster 4 — 교육 투자형 고단가 소비자

| 항목 | 내용 |
|------|------|
| **건당 평균** | 34,462원 (전 클러스터 최고) |
| **주요 업종** | 소매/유통(30%) · 생활서비스(13%) · **학문/교육(13.7%)** |
| **교육비 비율** | 타 클러스터 대비 최대 **28배** |

**특징**  
학원비·교재비 등 교육 관련 지출 비중이 두드러지며, 건당 지출액도 가장 높아 전반적으로 고단가 소비 성향을 보인다.

> 💬 교육에 과감히 투자하는 고단가 소비자

---

## 📊 클러스터 요약 비교

| 클러스터 | 명칭 | 건당가격 | 핵심 업종 | 주말비율 |
|---------|------|---------|---------|---------|
| **C0** | 외식·여가 중심 | 13,827원 | 음식·여가 | 42.5% |
| **C1** | 소액 쇼핑 중심 | 6,833원 | 소매/유통 | 29.9% |
| **C2** | 생활 전반 대량 | 29,374원 | 생활서비스 | 42.5% |
| **C3** | 건강관리 중심 | 21,150원 | 의료/건강 | 39.2% |
| **C4** | 교육 투자형 | 34,462원 | 학문/교육 | 42.3% |

> 💡 **공통점**: 소매/유통은 전 클러스터 공통 상위 카테고리 — 한국 소비의 기반은 역시 쇼핑!
